# Engineering Drawing Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B that reads engineering drawings, generates manufacturing
checklists, explains tolerances, and flags AS1100 compliance issues.

Runs fully offline on consumer hardware (T4 GPU). No cloud dependency.

In [ ]:
# Cell 1: Install dependencies and clone repo
!pip install unsloth[colab-new] trl datasets pandas pyarrow -q
!git clone https://github.com/govindrathore27/gemma4-engineering-diagrams.git /kaggle/working/repo 2>/dev/null || git -C /kaggle/working/repo pull
import sys, os
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Cell 2: Download AS1100 dataset from HuggingFace and generate training data
import os, json, random
from pathlib import Path

DATA_DIR    = Path("/kaggle/working/data/as1100")
TRAIN_JSONL = Path("/kaggle/working/drawing_train.jsonl")
EVAL_JSONL  = Path("/kaggle/working/drawing_eval.jsonl")

_TEMPLATES = [
    ("List the critical dimensions for a shaft drawing.",
     '{"dimensions": ["overall_length", "shaft_diameter", "keyway_width", "bore_diameter"]}'),
    ("What does a surface finish callout of Ra 1.6 mean?",
     "Ra 1.6 μm indicates a fine machined surface, suitable for moving parts requiring smooth contact."),
    ("Explain the difference between bilateral and unilateral tolerances.",
     "Bilateral tolerances allow variation in both directions (e.g., ±0.05 mm). "
     "Unilateral tolerances allow variation in one direction only (e.g., +0.00/−0.10 mm)."),
    ("What is a first-angle projection and when is it used?",
     "First-angle (European) projection places views as if the object is rotated away from the viewer. "
     "It is the standard in AS1100 Australian/British engineering drawings."),
    ("How do you verify a drawing conforms to AS1100 standard?",
     "Check the title block format, projection angle symbol, dimension style, surface finish "
     "notation, and tolerance format all comply with AS1100 Part 101."),
    ("What should be included in an engineering drawing title block?",
     '{"required": ["part_number", "title", "material", "scale", "drawn_by", "date", "revision"]}'),
    ("What is GD&T?",
     "GD&T (Geometric Dimensioning and Tolerancing) is a system for defining tolerances using symbols "
     "to specify form, orientation, location, and runout of features."),
    ("How do you read a tolerance of 25.00 ± 0.05?",
     "The nominal dimension is 25.00 mm. The part must be between 24.95 mm and 25.05 mm to pass inspection."),
]

if TRAIN_JSONL.exists():
    print("Training data already exists, skipping generation")
else:
    all_pairs = []

    # Try HuggingFace AS1100 dataset
    try:
        from huggingface_hub import snapshot_download
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        snapshot_download(
            repo_id="jcrzd/engineering-drawings-as1100",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
        from shared.data_pipeline.generate_qa_pairs import parse, save_jsonl
        for pf in (DATA_DIR / "data").glob("*.parquet"):
            pairs = parse("as1100_parquet", str(pf))
            all_pairs.extend(pairs)
            print(f"  {pf.name}: {len(pairs)} pairs")
    except Exception as e:
        print(f"[WARN] HuggingFace download failed: {e}")
        from shared.data_pipeline.generate_qa_pairs import save_jsonl

    print(f"Real QA pairs: {len(all_pairs)}")

    # Pad with synthetic templates to reach 1500
    while len(all_pairs) < 1500:
        q, a = random.choice(_TEMPLATES)
        all_pairs.append({"instruction": q, "input": "", "output": a})

    split_idx = int(len(all_pairs) * 0.8)
    save_jsonl(all_pairs[:split_idx], str(TRAIN_JSONL))
    save_jsonl(all_pairs[split_idx:], str(EVAL_JSONL))
    print(f"Generated {len(all_pairs)} pairs → {split_idx} train, {len(all_pairs)-split_idx} eval")

In [ ]:
# Cell 3: Train the Engineering Drawing Tutor model
os.environ["WANDB_DISABLED"] = "true"

from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="drawing",
    data_path=str(TRAIN_JSONL),
    output_dir="/kaggle/working/drawing_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 4: Evaluate
import json
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/drawing_adapter")
print("Model loaded")

eval_rows = [json.loads(l) for l in open(str(EVAL_JSONL), encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected = [r["output"] for r in eval_rows]

em = evaluate_batch(predictions, expected, metric="exact_match")
f1 = evaluate_batch(predictions, expected, metric="f1")
print(f"Exact Match: {em['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")

with open("/kaggle/working/eval_results.txt", "w") as f:
    f.write(f"Exact Match: {em['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 5: Export to GGUF
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/drawing_adapter",
    output_path="/kaggle/working/drawing_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 6: Demo inference
ctx = "Drawing: Shaft assembly, AS1100 standard. Key features: shaft diameter 25.00mm, keyway 6x6mm, length 120mm"

questions = [
    "List the critical dimensions for a shaft drawing.",
    "Create a manufacturing checklist for this engineering drawing.",
    "How do you verify a drawing conforms to AS1100 standard?",
    "What does a surface finish callout of Ra 1.6 mean?",
    "How do you read a tolerance of 25.00 ± 0.05?",
]

print("=== Engineering Drawing Tutor Demo ===\n")
for q in questions:
    answer = run(model, tokenizer, q, ctx)
    print(f"Q: {q}\nA: {answer}\n")

## Real-World Impact

Apprentice machinists in developing countries receive AS1100 drawings but have no
access to instructors who can explain GD&T symbols, tolerances, or compliance requirements.
This model functions as an offline pocket tutor.

**Datasets used:**
- AS1100 Engineering Drawings (HuggingFace jcrzd, CC-BY-4.0): 24 annotated drawings
- Synthetic QA templates: 10 curated AS1100/GD&T question-answer pairs
- Training: 1,200 QA pairs (24 gold + synthetic pad)